In [1]:
import os
import sys
import pickle
from pathlib import Path

from dotenv import load_dotenv
from neo4j import GraphDatabase

sys.path.insert(0, str(Path("..").resolve()))
load_dotenv("../.env")

True

In [2]:
from graph.embedder import Embedder
from graph.kin_rag import (
    StatelessCypherRetriever,
    CypherNativeRetriever,
    AblationConfig,
    AblationEvaluator,
)

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password123")

DATASET_NAME = "2wiki" # "musique" or "2wiki"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
embedder = Embedder(cache_path=Path("cache/embeddings_cache.pkl"))

[Embedder] Loaded 828 cached embeddings


In [3]:
with open(f"ground_truth-{DATASET_NAME}.pkl", "rb") as f:
    ground_truth = pickle.load(f)

print(f"Loaded {len(ground_truth)} questions for {DATASET_NAME}")
ground_truth[0]

Loaded 1000 questions for 2wiki


{'question_id': 'q_0',
 'question': 'Who is the mother of the director of film Polish-Russian War (Film)?',
 'answer': 'Małgorzata Braunek',
 'supporting_paragraphs': ['Polish-Russian War (film)__72bff114a8',
  'Xawery Żuławski__d99ca553fb']}

## Ablation Configurations

Run configs one by one — each takes ~1h for 1000 questions. Use `n_eval` to test on a subset first.

In [4]:
# retriever_cls = CypherNativeRetriever
retriever_cls = StatelessCypherRetriever

evaluator = AblationEvaluator(
    driver=driver,
    embedder=embedder,
    ground_truth=ground_truth,

    output_dir=f"ablation-results-{DATASET_NAME}",
    llm_model="gemini-2.5-flash",
    llm_judge=False,
    retriever_cls=retriever_cls,
    query_entity_cache_path=Path(f"cache/query_entities-{DATASET_NAME}.pkl"),
)

[Evaluator] Loaded 1000 cached query entities


In [5]:
configs = {
    # --- Baseline: no graph at all ---
    "naive_vector": AblationConfig(
        name="naive-vector",
        naive_vector=True,
        top_k_entities=30,
    ),

    # --- Ablation: remove query-awareness (uniform propagation) ---
    "no_qa": AblationConfig(
        name="no-query-aware",
        query_aware=False,
        max_steps=3,
        decay=0.7,
    ),

    # --- Vary max_steps (decay=0.7, query_aware=True) ---
    "steps_1": AblationConfig(name="steps-1", max_steps=1, decay=0.7),
    "steps_2": AblationConfig(name="steps-2", max_steps=2, decay=0.7),
    "steps_3": AblationConfig(name="steps-3", max_steps=3, decay=0.7),
    "steps_4": AblationConfig(name="steps-4", max_steps=4, decay=0.7),

    # --- Vary decay (max_steps=3, query_aware=True) ---
    "decay_03": AblationConfig(name="decay-0.3", max_steps=3, decay=0.3),
    "decay_05": AblationConfig(name="decay-0.5", max_steps=3, decay=0.5),
    "decay_07": AblationConfig(name="decay-0.7", max_steps=3, decay=0.7),
    "decay_09": AblationConfig(name="decay-0.9", max_steps=3, decay=0.9),
    "decay_10": AblationConfig(name="decay-1.0", max_steps=3, decay=1.0),
}

for c in configs.values():
    print(c.describe())

[naive-vector] naive vector search, top_k=30
[no-query-aware] steps=3, decay=0.7, uniform, threshold=0.01
[steps-1] steps=1, decay=0.7, query-aware, threshold=0.01
[steps-2] steps=2, decay=0.7, query-aware, threshold=0.01
[steps-3] steps=3, decay=0.7, query-aware, threshold=0.01
[steps-4] steps=4, decay=0.7, query-aware, threshold=0.01
[decay-0.3] steps=3, decay=0.3, query-aware, threshold=0.01
[decay-0.5] steps=3, decay=0.5, query-aware, threshold=0.01
[decay-0.7] steps=3, decay=0.7, query-aware, threshold=0.01
[decay-0.9] steps=3, decay=0.9, query-aware, threshold=0.01
[decay-1.0] steps=3, decay=1.0, query-aware, threshold=0.01


## Run Experiments

Run one config at a time. Set `n_eval` to a small number (e.g. 50) for a quick sanity check, then `None` for the full dataset.

In [6]:
N_EVAL = 500  # set to None for full dataset

In [7]:
evaluator.run_config(configs["naive_vector"], n_eval=N_EVAL, max_workers=10)


  [naive-vector] naive vector search, top_k=30
  workers=10


naive-vector: 100%|██████████| 500/500 [01:02<00:00,  8.05it/s, EM=0.350, F1=0.399, ok=500]


  [naive-vector] 500 questions in 62s
  EM:  0.3500
  F1:  0.3990
  Saved: ablation-results-2wiki/naive-vector.csv


RunResult(config=AblationConfig(name='naive-vector', max_steps=3, decay=0.7, query_aware=True, naive_vector=True, top_k_paths=30, top_k_entities=30, seed_vector_fallback_k=5, resource_threshold=0.01), df=    question_id                                           question  \
0           q_0  Who is the mother of the director of film Poli...   
1           q_1  Which film came out first, Blind Shaft or The ...   
2           q_2  When did John V, Prince Of Anhalt-Zerbst's fat...   
3           q_3  What is the award that the director of film We...   
4           q_4  Where was the director of film Ronnie Rocket b...   
..          ...                                                ...   
495       q_495  Do both films Fig Leaves and Boogeyman (Film) ...   
496       q_496  Which film has the director who is older, Road...   
497       q_497  Who is Prince Harald Of Schaumburg-Lippe's mat...   
498       q_498  Where was the place of burial of Gjon Kastriot...   
499       q_499           

In [8]:
evaluator.run_config(configs["no_qa"], n_eval=N_EVAL, max_workers=10)


  [no-query-aware] steps=3, decay=0.7, uniform, threshold=0.01
  workers=10


no-query-aware: 100%|██████████| 500/500 [01:07<00:00,  7.42it/s, EM=0.450, F1=0.518, ok=500]


  [no-query-aware] 500 questions in 67s
  EM:  0.4500
  F1:  0.5176
  Saved: ablation-results-2wiki/no-query-aware.csv


RunResult(config=AblationConfig(name='no-query-aware', max_steps=3, decay=0.7, query_aware=False, naive_vector=False, top_k_paths=30, top_k_entities=30, seed_vector_fallback_k=5, resource_threshold=0.01), df=    question_id                                           question  \
0           q_0  Who is the mother of the director of film Poli...   
1           q_1  Which film came out first, Blind Shaft or The ...   
2           q_2  When did John V, Prince Of Anhalt-Zerbst's fat...   
3           q_3  What is the award that the director of film We...   
4           q_4  Where was the director of film Ronnie Rocket b...   
..          ...                                                ...   
495       q_495  Do both films Fig Leaves and Boogeyman (Film) ...   
496       q_496  Which film has the director who is older, Road...   
497       q_497  Who is Prince Harald Of Schaumburg-Lippe's mat...   
498       q_498  Where was the place of burial of Gjon Kastriot...   
499       q_499       

In [9]:
evaluator.run_config(configs["steps_1"], n_eval=N_EVAL, max_workers=10)


  [steps-1] steps=1, decay=0.7, query-aware, threshold=0.01
  workers=10


steps-1: 100%|██████████| 500/500 [00:44<00:00, 11.32it/s, EM=0.330, F1=0.384, ok=500]



  [steps-1] 500 questions in 44s
  EM:  0.3300
  F1:  0.3842
  Saved: ablation-results-2wiki/steps-1.csv


RunResult(config=AblationConfig(name='steps-1', max_steps=1, decay=0.7, query_aware=True, naive_vector=False, top_k_paths=30, top_k_entities=30, seed_vector_fallback_k=5, resource_threshold=0.01), df=    question_id                                           question  \
0           q_0  Who is the mother of the director of film Poli...   
1           q_1  Which film came out first, Blind Shaft or The ...   
2           q_2  When did John V, Prince Of Anhalt-Zerbst's fat...   
3           q_3  What is the award that the director of film We...   
4           q_4  Where was the director of film Ronnie Rocket b...   
..          ...                                                ...   
495       q_495  Do both films Fig Leaves and Boogeyman (Film) ...   
496       q_496  Which film has the director who is older, Road...   
497       q_497  Who is Prince Harald Of Schaumburg-Lippe's mat...   
498       q_498  Where was the place of burial of Gjon Kastriot...   
499       q_499            Who

In [10]:
evaluator.run_config(configs["steps_2"], n_eval=N_EVAL, max_workers=10)


  [steps-2] steps=2, decay=0.7, query-aware, threshold=0.01
  workers=10


steps-2: 100%|██████████| 500/500 [00:52<00:00,  9.60it/s, EM=0.506, F1=0.572, ok=500]


  [steps-2] 500 questions in 52s
  EM:  0.5060
  F1:  0.5718
  Saved: ablation-results-2wiki/steps-2.csv


RunResult(config=AblationConfig(name='steps-2', max_steps=2, decay=0.7, query_aware=True, naive_vector=False, top_k_paths=30, top_k_entities=30, seed_vector_fallback_k=5, resource_threshold=0.01), df=    question_id                                           question  \
0           q_0  Who is the mother of the director of film Poli...   
1           q_1  Which film came out first, Blind Shaft or The ...   
2           q_2  When did John V, Prince Of Anhalt-Zerbst's fat...   
3           q_3  What is the award that the director of film We...   
4           q_4  Where was the director of film Ronnie Rocket b...   
..          ...                                                ...   
495       q_495  Do both films Fig Leaves and Boogeyman (Film) ...   
496       q_496  Which film has the director who is older, Road...   
497       q_497  Who is Prince Harald Of Schaumburg-Lippe's mat...   
498       q_498  Where was the place of burial of Gjon Kastriot...   
499       q_499            Who

In [11]:
evaluator.run_config(configs["steps_3"], n_eval=N_EVAL, max_workers=10)


  [steps-3] steps=3, decay=0.7, query-aware, threshold=0.01
  workers=10


steps-3: 100%|██████████| 500/500 [00:51<00:00,  9.62it/s, EM=0.490, F1=0.559, ok=500]


  [steps-3] 500 questions in 52s
  EM:  0.4900
  F1:  0.5591
  Saved: ablation-results-2wiki/steps-3.csv


RunResult(config=AblationConfig(name='steps-3', max_steps=3, decay=0.7, query_aware=True, naive_vector=False, top_k_paths=30, top_k_entities=30, seed_vector_fallback_k=5, resource_threshold=0.01), df=    question_id                                           question  \
0           q_0  Who is the mother of the director of film Poli...   
1           q_1  Which film came out first, Blind Shaft or The ...   
2           q_2  When did John V, Prince Of Anhalt-Zerbst's fat...   
3           q_3  What is the award that the director of film We...   
4           q_4  Where was the director of film Ronnie Rocket b...   
..          ...                                                ...   
495       q_495  Do both films Fig Leaves and Boogeyman (Film) ...   
496       q_496  Which film has the director who is older, Road...   
497       q_497  Who is Prince Harald Of Schaumburg-Lippe's mat...   
498       q_498  Where was the place of burial of Gjon Kastriot...   
499       q_499            Who

In [12]:
evaluator.run_config(configs["steps_4"], n_eval=N_EVAL, max_workers=10)


  [steps-4] steps=4, decay=0.7, query-aware, threshold=0.01
  workers=10


steps-4: 100%|██████████| 500/500 [01:13<00:00,  6.82it/s, EM=0.506, F1=0.575, ok=500]


  [steps-4] 500 questions in 74s
  EM:  0.5060
  F1:  0.5751
  Saved: ablation-results-2wiki/steps-4.csv


RunResult(config=AblationConfig(name='steps-4', max_steps=4, decay=0.7, query_aware=True, naive_vector=False, top_k_paths=30, top_k_entities=30, seed_vector_fallback_k=5, resource_threshold=0.01), df=    question_id                                           question  \
0           q_0  Who is the mother of the director of film Poli...   
1           q_1  Which film came out first, Blind Shaft or The ...   
2           q_2  When did John V, Prince Of Anhalt-Zerbst's fat...   
3           q_3  What is the award that the director of film We...   
4           q_4  Where was the director of film Ronnie Rocket b...   
..          ...                                                ...   
495       q_495  Do both films Fig Leaves and Boogeyman (Film) ...   
496       q_496  Which film has the director who is older, Road...   
497       q_497  Who is Prince Harald Of Schaumburg-Lippe's mat...   
498       q_498  Where was the place of burial of Gjon Kastriot...   
499       q_499            Who

In [13]:
evaluator.run_config(configs["decay_03"], n_eval=N_EVAL, max_workers=10)


  [decay-0.3] steps=3, decay=0.3, query-aware, threshold=0.01
  workers=10


decay-0.3: 100%|██████████| 500/500 [00:52<00:00,  9.50it/s, EM=0.502, F1=0.572, ok=500]


  [decay-0.3] 500 questions in 53s
  EM:  0.5020
  F1:  0.5717
  Saved: ablation-results-2wiki/decay-0.3.csv


RunResult(config=AblationConfig(name='decay-0.3', max_steps=3, decay=0.3, query_aware=True, naive_vector=False, top_k_paths=30, top_k_entities=30, seed_vector_fallback_k=5, resource_threshold=0.01), df=    question_id                                           question  \
0           q_0  Who is the mother of the director of film Poli...   
1           q_1  Which film came out first, Blind Shaft or The ...   
2           q_2  When did John V, Prince Of Anhalt-Zerbst's fat...   
3           q_3  What is the award that the director of film We...   
4           q_4  Where was the director of film Ronnie Rocket b...   
..          ...                                                ...   
495       q_495  Do both films Fig Leaves and Boogeyman (Film) ...   
496       q_496  Which film has the director who is older, Road...   
497       q_497  Who is Prince Harald Of Schaumburg-Lippe's mat...   
498       q_498  Where was the place of burial of Gjon Kastriot...   
499       q_499            W

In [14]:
evaluator.run_config(configs["decay_05"], n_eval=N_EVAL, max_workers=10)


  [decay-0.5] steps=3, decay=0.5, query-aware, threshold=0.01
  workers=10


decay-0.5: 100%|██████████| 500/500 [00:55<00:00,  8.97it/s, EM=0.484, F1=0.551, ok=500]


  [decay-0.5] 500 questions in 56s
  EM:  0.4840
  F1:  0.5513
  Saved: ablation-results-2wiki/decay-0.5.csv


RunResult(config=AblationConfig(name='decay-0.5', max_steps=3, decay=0.5, query_aware=True, naive_vector=False, top_k_paths=30, top_k_entities=30, seed_vector_fallback_k=5, resource_threshold=0.01), df=    question_id                                           question  \
0           q_0  Who is the mother of the director of film Poli...   
1           q_1  Which film came out first, Blind Shaft or The ...   
2           q_2  When did John V, Prince Of Anhalt-Zerbst's fat...   
3           q_3  What is the award that the director of film We...   
4           q_4  Where was the director of film Ronnie Rocket b...   
..          ...                                                ...   
495       q_495  Do both films Fig Leaves and Boogeyman (Film) ...   
496       q_496  Which film has the director who is older, Road...   
497       q_497  Who is Prince Harald Of Schaumburg-Lippe's mat...   
498       q_498  Where was the place of burial of Gjon Kastriot...   
499       q_499            W

In [15]:
evaluator.run_config(configs["decay_07"], n_eval=N_EVAL, max_workers=10)


  [decay-0.7] steps=3, decay=0.7, query-aware, threshold=0.01
  workers=10


decay-0.7: 100%|██████████| 500/500 [00:55<00:00,  8.95it/s, EM=0.490, F1=0.558, ok=500]


  [decay-0.7] 500 questions in 56s
  EM:  0.4900
  F1:  0.5581
  Saved: ablation-results-2wiki/decay-0.7.csv


RunResult(config=AblationConfig(name='decay-0.7', max_steps=3, decay=0.7, query_aware=True, naive_vector=False, top_k_paths=30, top_k_entities=30, seed_vector_fallback_k=5, resource_threshold=0.01), df=    question_id                                           question  \
0           q_0  Who is the mother of the director of film Poli...   
1           q_1  Which film came out first, Blind Shaft or The ...   
2           q_2  When did John V, Prince Of Anhalt-Zerbst's fat...   
3           q_3  What is the award that the director of film We...   
4           q_4  Where was the director of film Ronnie Rocket b...   
..          ...                                                ...   
495       q_495  Do both films Fig Leaves and Boogeyman (Film) ...   
496       q_496  Which film has the director who is older, Road...   
497       q_497  Who is Prince Harald Of Schaumburg-Lippe's mat...   
498       q_498  Where was the place of burial of Gjon Kastriot...   
499       q_499            W

In [16]:
evaluator.run_config(configs["decay_09"], n_eval=N_EVAL, max_workers=10)


  [decay-0.9] steps=3, decay=0.9, query-aware, threshold=0.01
  workers=10


decay-0.9: 100%|██████████| 500/500 [00:51<00:00,  9.69it/s, EM=0.476, F1=0.546, ok=500]


  [decay-0.9] 500 questions in 52s
  EM:  0.4760
  F1:  0.5459
  Saved: ablation-results-2wiki/decay-0.9.csv


RunResult(config=AblationConfig(name='decay-0.9', max_steps=3, decay=0.9, query_aware=True, naive_vector=False, top_k_paths=30, top_k_entities=30, seed_vector_fallback_k=5, resource_threshold=0.01), df=    question_id                                           question  \
0           q_0  Who is the mother of the director of film Poli...   
1           q_1  Which film came out first, Blind Shaft or The ...   
2           q_2  When did John V, Prince Of Anhalt-Zerbst's fat...   
3           q_3  What is the award that the director of film We...   
4           q_4  Where was the director of film Ronnie Rocket b...   
..          ...                                                ...   
495       q_495  Do both films Fig Leaves and Boogeyman (Film) ...   
496       q_496  Which film has the director who is older, Road...   
497       q_497  Who is Prince Harald Of Schaumburg-Lippe's mat...   
498       q_498  Where was the place of burial of Gjon Kastriot...   
499       q_499            W

In [17]:
evaluator.run_config(configs["decay_10"], n_eval=N_EVAL, max_workers=10)


  [decay-1.0] steps=3, decay=1.0, query-aware, threshold=0.01
  workers=10


decay-1.0: 100%|██████████| 500/500 [00:51<00:00,  9.76it/s, EM=0.504, F1=0.574, ok=500]


  [decay-1.0] 500 questions in 51s
  EM:  0.5040
  F1:  0.5740
  Saved: ablation-results-2wiki/decay-1.0.csv


RunResult(config=AblationConfig(name='decay-1.0', max_steps=3, decay=1.0, query_aware=True, naive_vector=False, top_k_paths=30, top_k_entities=30, seed_vector_fallback_k=5, resource_threshold=0.01), df=    question_id                                           question  \
0           q_0  Who is the mother of the director of film Poli...   
1           q_1  Which film came out first, Blind Shaft or The ...   
2           q_2  When did John V, Prince Of Anhalt-Zerbst's fat...   
3           q_3  What is the award that the director of film We...   
4           q_4  Where was the director of film Ronnie Rocket b...   
..          ...                                                ...   
495       q_495  Do both films Fig Leaves and Boogeyman (Film) ...   
496       q_496  Which film has the director who is older, Road...   
497       q_497  Who is Prince Harald Of Schaumburg-Lippe's mat...   
498       q_498  Where was the place of burial of Gjon Kastriot...   
499       q_499            W

## Summary

In [18]:
summary = evaluator.summary()
summary

,config,n,em,f1,avg_prompt_tok,avg_total_tok,avg_paths,avg_entities,avg_retrieval_s,avg_generation_s
10,steps-4,500,0.506,0.575090,4749.170,4798.024,18.348,28.802,0.573790,0.888091
4,decay-1.0,500,0.504,0.574023,4621.198,4669.478,18.718,28.398,0.172997,0.837286
8,steps-2,500,0.506,0.571819,3616.544,3665.282,16.236,24.342,0.061446,0.874552
0,decay-0.3,500,0.502,0.571656,3683.944,3732.570,16.580,24.722,0.207879,0.827989
9,steps-3,500,0.490,0.559089,4504.914,4554.460,17.996,28.298,0.181510,0.858331
2,decay-0.7,500,0.490,0.558089,4504.914,4554.432,17.996,28.298,0.225931,0.888553
1,decay-0.5,500,0.484,0.551264,4242.138,4290.594,17.424,27.332,0.246112,0.861428
3,decay-0.9,500,0.476,0.545851,4586.794,4635.480,18.460,28.394,0.166566,0.853402
6,no-query-aware,500,0.450,0.517630,5137.138,5186.114,19.060,28.566,0.423899,0.901374
5,naive-vector,500,0.350,0.398959,1781.732,1829.576,0.000,30.000,0.320441,0.884132


In [ ]:
driver.close()